In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import matplotlib.pyplot as plt

from torch import optim
from stochinter.models import UnconditionalModel, ConditionalModel
from stochinter.utils import solve_ode
from sklearn.datasets import make_moons
import numpy as np

In [ ]:
# parameters
SEED = 42
BATCH_SIZE = 1024
EPOCHS = 1_000
LEARNING_RATE = 3e-3

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
batch_size = BATCH_SIZE
epochs = EPOCHS

net_unconditional = UnconditionalModel()
net_conditional = ConditionalModel()

opt_uncon = optim.Adam(net_unconditional.parameters(), lr=LEARNING_RATE)
opt_con = optim.Adam(net_conditional.parameters(), lr=LEARNING_RATE)

print("Training of models")

for epoch in range(epochs):
    x0 = torch.randn(batch_size, 2)
    
    x1_np, omega_np = make_moons(n_samples=batch_size, noise=0.05)
    x1 = torch.tensor((x1_np - np.array([0.5, 0.25])) * 2.0, dtype=torch.float32)
    omega = torch.tensor(omega_np).unsqueeze(1)
    
    z = torch.randn(batch_size, 2)*0.1
    t = torch.rand(batch_size, 1)
    
    # Interpolation
    x_t = (1 - t) * x0 + t * x1 + torch.sqrt(2*t*(1-t))*z 
    
    # Target for vector field
    v_target = x1 - x0 + z*(1-2*t) / torch.sqrt(2*t*(1-t))

    opt_uncon.zero_grad()
    v_pred_uncon = net_unconditional(x_t, t)
    loss_uncon = torch.mean((v_pred_uncon - v_target)**2)
    loss_uncon.backward()
    opt_uncon.step()
    
    opt_con.zero_grad()
    v_pred_con = net_conditional(x_t, omega, t)
    loss_con = torch.mean((v_pred_con - v_target)**2)
    loss_con.backward()
    opt_con.step()
    
    if (epoch+1) % 500 == 0:
        print(f"Epoch {epoch+1:4d} | Scalar loss: {loss_uncon.item():.4f} | conect loss: {loss_con.item():.4f}")

In [ ]:
print("Running inference...")

n_test = 2500
x_start = torch.randn(n_test, 2)
steps = 30

half_size = n_test // 2
classes = torch.cat([
    torch.zeros(half_size, dtype=torch.long), 
    torch.ones(n_test - half_size, dtype=torch.long)
]).unsqueeze(1)


@torch.no_grad()
def velocity_uncon(x, t):
    return net_unconditional(x, t)

@torch.no_grad()
def velocity_con(x, t):
    return net_conditional(x, classes, t)


x_gen_uncon = solve_ode(x_start, velocity_uncon, steps=steps)
x_gen_con = solve_ode(x_start, velocity_con, steps=steps)

print("Inference done!")

In [ ]:
x_true_np, _ = make_moons(n_samples=batch_size, noise=0.05)
x_true = torch.tensor((x1_np - np.array([0.5, 0.25])) * 2.0, dtype=torch.float32)

x_start_np = x_start.cpu().numpy()
x_true_np = x_true.cpu().numpy()
x_uncon_np = x_gen_uncon.cpu().numpy()
x_con_np = x_gen_con.cpu().numpy()

def plot_scatter(ax, data, title):
    ax.scatter(data[:, 0], data[:, 1], s=4, alpha=0.3, color='black')
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.5)

In [ ]:
fig1, axes1 = plt.subplots(1, 2, figsize=(10, 5))

plot_scatter(axes1[0], x_start_np, "Initial Noise ($x_0$)")
plot_scatter(axes1[1], x_true_np, "Target (True Ring)")

fig1.tight_layout()
plt.show()

In [ ]:
fig2, axes2 = plt.subplots(1, 2, figsize=(10, 5))

plot_scatter(axes2[0], x_uncon_np, "Generated (UnconditionalNet)")
plot_scatter(axes2[1], x_con_np, "Generated (ConditionalNet)")

fig2.tight_layout()
fig2.savefig('experiment1.png', dpi=600)
plt.show()

In [ ]:
from scipy.stats import wasserstein_distance_nd
print(wasserstein_distance_nd(x_true_np, x_uncon_np))
print(wasserstein_distance_nd(x_true_np, x_con_np))